## Generation of transit windows:
In this notebook we will show how to calculate transit windows using `PTO`, and how to use those windows to plot observing conditions based on observatory.


## 0. Initial setup
Lets create filtered table for our dataset sample. Here, we will use only a single target filter, focusing on WASP-76b.

In [1]:
import os
os.chdir('../')

from PTO.database.NASA_exoplanet_archive import NASA_Exoplanet_Archive_CompositeDefault

Catalog = NASA_Exoplanet_Archive_CompositeDefault()
Catalog.load_API_table()
Catalog.table = Catalog.table[Catalog.table['Planet.Name'] == 'WASP-76 b']

  INFO     | Trying to load NASA Exoplanet Archive Composite table
  INFO     | Filename: ./saved_files/CatalogComposite.pkl
  INFO     | Accessing NASA Exoplanet Archive
  INFO     | Fetching table
  INFO     | Table fetched successfully
  WARNING  |     Droping all values without errorbars. To instead replace the errorbars with 0 change Catalogs "drop_mode" key to "replace"
  INFO     | Converting Earth and Jupiter Radius units in the catalog
  INFO     | Checking for inclination and impact parameters values
  INFO     | Calculation of T_14 and related values
  INFO     | File saved succesfully in:
  INFO     |     /media/chamaeleontis/Observatory_main/Code/observations_transits/PTO/saved_files/CatalogComposite.pkl


## 1. Calculation of transit windows
Transit windows are handled using the `PTO.transits.windows.Windows` class. This class needs as its input the filtered table (`.table`), observing period (`.observing_period`) during which to calculate the windows, baseline (`.baseline`) and whether we observe a standard or large program (`.large_program`). 

We will select a observing period by ESO for P115, with no large program flag. Baseline has a default function, which will be run for each planet separately, if None (default value). If a Quantity is given, all windows are provided with given baseline. This is useful for emission observation, which are offset by given phase.

In [2]:
from PTO.transits.windows import Windows
Transits = Windows(
    table = Catalog.table,
    observing_period = 'ESO.115',
    large_program= False
)

  INFO     | =========================
  INFO     | Set observing period:
  INFO     |     2025-04-01 12:00:00.000
  INFO     |     2025-10-01 12:00:00.000
  INFO     | =========================
  INFO     | About to calculate event midpoints for 1 planets


## 2. Calculate observability of each window
Because the center of transit windows is independent of location, the previous class has no information about observability. In order to calculate observability for given window, we need to define location.

In `PTO`, you can use multiple formats for location. The default, is to use the `PTO.telescopes.Telescope` class, which holds the information about the location, size and available instruments for given telescope. A convenience function to print out all available option is given.


In [3]:
import PTO.telescopes.telescopes as tel
tel.print_all_telescopes()

  PRINT    | Very Large Telescope (VLT) 4-Unit Telescope | 16.4 m | Operational: True
  PRINT    |     ['ESPRESSO_4UT']
  PRINT    | Very Large Telescope (VLT) Unit Telescope | 8.2 m | Operational: True
  PRINT    |     ['ESPRESSO', 'UVES']
  PRINT    | Gemini North Telescope | 8.1 m | Operational: True
  PRINT    |     ['MAROON-X']
  PRINT    | Lowell Discovery Telescope | 4.3 m | Operational: True
  PRINT    |     ['EXPRES']
  PRINT    | ESO 3.6m Telescope (La Silla) | 3.6 m | Operational: True
  PRINT    |     ['HARPS', 'NIRPS']
  PRINT    | Canada-France-Hawaii Telescope (CFHT) | 3.6 m | Operational: True
  PRINT    |     ['SPIRou']
  PRINT    | Telescopio Nazionale Galileo (TNG) | 3.58 m | Operational: True
  PRINT    |     ['GIANO', 'HARPS-N']
  PRINT    | Calar Alto Observatory | 3.5 m | Operational: True
  PRINT    |     ['CARMENES']
  PRINT    | Haute-Provence Observatory | 1.93 m | Operational: True
  PRINT    |     ['SOPHIE']
  PRINT    | Swiss Euler Telescope | 1.2 m | Oper

Lets use the VLT with standard 1UT mode (4_UT mode is a separate instance of the Telescope class, as the utility is divided).
To do so:

In [4]:
Transits.generate_observability(
    location= tel.VLT,
)

TypeError: Windows.generate_observability() missing 1 required positional argument: 'partial'

Recommendation: Always check the values used for the planner. The key 5 arguments are:
- 'Position.RightAscension'
- 'Position.Declination'
- 'Planet.TransitMidpoint'
- 'Planet.Period'
- 'Planet.TransitDuration'

A `print_ephemeris()` method is provided to double-check all ephemeris used in the sample for convenience.
